# LMSYS — Qwen2.5-7B + QLoRA — **INFERENCE** notebook (2 of 2)

Loads the LoRA adapter trained by `lmsys-qwen7b-qlora-train.ipynb`, predicts the hidden test set,
and writes `submission.csv`. **This is the notebook you submit to the competition.**

### Before running

1. Run the training notebook to completion and *Save Version*.
2. Here: **Add Input → Notebook Output →** the training notebook.
3. **Add Input → Competitions →** LLM Classification Finetuning.
4. Accelerator **GPU T4 x2**, Internet **ON** (the 15 GB base model is downloaded fresh — only the
   small adapter comes from the training run).

### Budget

~1.9 samples/s on one T4 at 768 tokens, so the 25,000-row hidden test set takes **~3.5–4h** —
inside Kaggle's 9h submission limit. The public `test.csv` you see interactively has only 3 rows;
the real set appears only during the submission re-run.

Preprocessing here is **byte-identical** to training. Any drift between the two silently destroys the
score, so the formatting functions are copied verbatim rather than re-derived.

---
# 1 · Environment

In [ ]:
import importlib.util
import importlib.metadata
import subprocess
import sys


def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return None


if importlib.util.find_spec("torchao") is not None:
    _tv = _version("torchao")
    try:
        from packaging.version import parse as _parse
        _too_old = _tv is not None and _parse(_tv) < _parse("0.16.0")
    except Exception:
        _too_old = True
    if _too_old:
        print(f"torchao {_tv} too old for peft -> upgrading ...")
        _pip("-U", "torchao>=0.16.0")
        if _version("torchao") == _tv:
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                           check=False)

for _pkg in ("peft", "bitsandbytes", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        _pip(_pkg)

import gc
import json
import math
import os
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
import peft
from peft import PeftModel

T0 = time.time()


def fmt_time(s):
    s = int(max(0, s))
    h, r = divmod(s, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h {m:02d}m {s:02d}s"


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Set Accelerator -> GPU T4 x2.")

DEVICE = torch.device("cuda")
print("torch/transformers/peft:", torch.__version__, transformers.__version__, peft.__version__)
print("GPU:", torch.cuda.get_device_name(0))

---
# 2 · Locate the adapter and the data

In [ ]:
CLASS_NAMES = ["winner_model_a", "winner_model_b", "winner_tie"]


def find_dir(required_file, extra=()):
    root = "/kaggle/input"
    candidates = list(extra)
    if os.path.isdir(root):
        print("Contents of /kaggle/input:", sorted(os.listdir(root)))
        for d in sorted(os.listdir(root)):
            sub = os.path.join(root, d)
            candidates.append(sub)
            if os.path.isdir(sub):
                for s in sorted(os.listdir(sub)):
                    ss = os.path.join(sub, s)
                    if os.path.isdir(ss):
                        candidates.append(ss)
                        for t in sorted(os.listdir(ss)):
                            tt = os.path.join(ss, t)
                            if os.path.isdir(tt):
                                candidates.append(tt)
    for c in candidates:
        if os.path.exists(os.path.join(c, required_file)):
            return c
    raise FileNotFoundError(f"{required_file} not found. Searched:\n  " + "\n  ".join(candidates))


ADAPTER_DIR = find_dir("adapter_config.json", extra=["/kaggle/working/qwen_lora_adapter"])
DATA_DIR = find_dir("test.csv", extra=["/kaggle/input/llm-classification-finetuning",
                                       "/kaggle/input/competitions/llm-classification-finetuning"])
print("\nadapter :", ADAPTER_DIR)
print("data    :", DATA_DIR)

RUN_INFO = {}
_info = os.path.join(ADAPTER_DIR, "run_info.json")
if os.path.exists(_info):
    RUN_INFO = json.load(open(_info))
    print("\nrun_info from training:")
    for k, v in RUN_INFO.items():
        print(f"  {k:20s} {v}")

BASE_MODEL = RUN_INFO.get("base_model", "Qwen/Qwen2.5-7B-Instruct")
MAX_LENGTH = int(RUN_INFO.get("max_length", 768))
PROMPT_CHARS = int(RUN_INFO.get("prompt_chars", 600))
RESPONSE_CHARS = int(RUN_INFO.get("response_chars", 1000))
EVAL_BATCH_SIZE = 8

print(f"\nbase model {BASE_MODEL} | max_length {MAX_LENGTH}")

test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print("test.csv:", test_df.shape)
if len(test_df) < 100:
    print("NOTE: this is the small public placeholder. The real ~25k test set is only "
          "mounted during the competition submission re-run.")

---
# 3 · Preprocessing — identical to training

In [ ]:
def parse_list(x):
    if isinstance(x, list):
        items = x
    elif not isinstance(x, str):
        return [""]
    else:
        items = None
        for loader in (json.loads, __import__("ast").literal_eval):
            try:
                items = loader(x)
                break
            except Exception:
                continue
        if items is None:
            return [""]
    if not isinstance(items, (list, tuple)):
        items = [items]
    return ["" if v is None else str(v) for v in items] or [""]


def head_tail(text, budget, head_frac=0.55):
    if len(text) <= budget:
        return text
    head = int(budget * head_frac)
    tail = budget - head
    return text[:head] + "\n...[truncated]...\n" + text[-tail:]


def build_text(row):
    prompts = parse_list(row["prompt"])
    resp_a = parse_list(row["response_a"])
    resp_b = parse_list(row["response_b"])
    n = max(len(prompts), len(resp_a), len(resp_b))

    def get(lst, i):
        return lst[i] if i < len(lst) else ""

    turns = max(1, min(n, 3))
    pb = max(150, PROMPT_CHARS // turns)
    rb = max(250, RESPONSE_CHARS // turns)

    parts = []
    for i in range(min(n, turns)):
        p = head_tail(get(prompts, i), pb)
        a = head_tail(get(resp_a, i), rb)
        b = head_tail(get(resp_b, i), rb)
        prefix = f"## Round {i + 1}\n" if turns > 1 else ""
        parts.append(f"{prefix}Prompt: {p}\n\nResponse A: {a}\n\nResponse B: {b}")

    body = "\n\n".join(parts)
    return (
        "You are judging which chatbot response a human would prefer.\n\n"
        f"{body}\n\n"
        "Which response is better: A, B, or a tie?"
    )


test_df["text"] = test_df.apply(build_text, axis=1)
print("built inputs;  example:\n")
print(test_df.text.iloc[0][:600])

---
# 4 · Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"          # decoder classifiers pool at the first pad position

texts = test_df.text.tolist()
test_ids = []
for i in tqdm(range(0, len(texts), 512), desc="tokenize test"):
    enc = tokenizer(texts[i:i + 512], truncation=True, max_length=MAX_LENGTH,
                    padding=False, add_special_tokens=True)
    test_ids.extend(enc["input_ids"])

_lens = np.array([len(x) for x in test_ids])
print(f"token length: mean {_lens.mean():.0f} | p95 {np.percentile(_lens, 95):.0f} "
      f"| at cap {(_lens >= MAX_LENGTH).mean():.1%}")


class TextDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "idx": i}


class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        maxlen = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), maxlen), self.pad_id, dtype=torch.long)
        msk = torch.zeros((len(batch), maxlen), dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[i, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
            msk[i, :n] = 1
        return {"input_ids": ids, "attention_mask": msk,
                "idx": torch.tensor([b["idx"] for b in batch], dtype=torch.long)}


class LengthGroupedBatchSampler(Sampler):
    """Sort by length so dynamic padding wastes as little compute as possible.

    Row order is restored afterwards via the carried `idx`, so predictions still line up with test_df.
    """

    def __init__(self, lengths, batch_size):
        self.order = np.argsort(np.asarray(lengths), kind="stable")
        self.batch_size = batch_size

    def __iter__(self):
        for i in range(0, len(self.order), self.batch_size):
            yield self.order[i:i + self.batch_size].tolist()

    def __len__(self):
        return math.ceil(len(self.order) / self.batch_size)


test_loader = DataLoader(
    TextDataset(test_ids),
    batch_sampler=LengthGroupedBatchSampler(_lens, EVAL_BATCH_SIZE),
    collate_fn=PadCollator(tokenizer.pad_token_id),
    num_workers=2, pin_memory=True,
)

---
# 5 · Load base model + adapter

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,     # fp16: T4 is sm_75, no bf16
)

t0 = time.time()
model_kwargs = dict(num_labels=3, quantization_config=bnb_config,
                    device_map={"": 0}, trust_remote_code=True)
try:
    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, dtype=torch.float16, **model_kwargs)
except TypeError:
    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, **model_kwargs)

base.config.pad_token_id = tokenizer.pad_token_id
base.config.use_cache = False

model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
model.eval()

print(f"loaded in {fmt_time(time.time() - t0)}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# The score head must have come from the adapter, not a fresh random init.
_score = [n for n, _ in model.named_parameters() if "score" in n and "modules_to_save" in n]
print("score head restored from adapter:", bool(_score), _score[:2])
if not _score:
    print("WARNING: no saved score head found in the adapter. Predictions will be random. "
          "Check that the training notebook used modules_to_save=['score'].")


def make_autocast():
    try:
        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    except (TypeError, AttributeError):
        return torch.cuda.amp.autocast()

---
# 6 · Predict

In [ ]:
probs = np.zeros((len(test_ids), 3), dtype=np.float64)
t_inf = time.time()

with torch.no_grad():
    for batch in tqdm(test_loader, desc="inference"):
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        msk = batch["attention_mask"].to(DEVICE, non_blocking=True)
        with make_autocast():
            logits = model(input_ids=ids, attention_mask=msk).logits
        p = torch.softmax(logits.float(), -1).cpu().numpy()
        probs[batch["idx"].numpy()] = p          # scatter back to original row order

INFER_SEC = time.time() - t_inf
probs = probs / probs.sum(1, keepdims=True)

print(f"\ninference: {fmt_time(INFER_SEC)} for {len(test_ids):,} rows "
      f"({len(test_ids) / max(1e-9, INFER_SEC):.2f} rows/s)")
print("mean probs: " + ", ".join(f"{c}={v:.4f}" for c, v in zip(CLASS_NAMES, probs.mean(0))))

---
# 7 · Submission

In [ ]:
submission = test_df[["id"]].copy()
submission[CLASS_NAMES] = probs

assert list(submission.columns) == ["id"] + CLASS_NAMES, submission.columns.tolist()
assert len(submission) == len(test_df)
assert submission["id"].is_unique
assert submission[CLASS_NAMES].notna().all().all()
assert (submission[CLASS_NAMES] >= 0).all().all()
assert np.allclose(submission[CLASS_NAMES].sum(axis=1), 1.0, atol=1e-5)

submission.to_csv("submission.csv", index=False)

print("submission.csv written", submission.shape)
display(submission.head())
print("\nall checks passed: columns, unique ids, non-negative, rows sum to 1")
print(f"\nTotal notebook runtime: {fmt_time(time.time() - T0)}")